In [1]:
import pandas as pd
import numpy as np
import itertools
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.inspection import permutation_importance

In [2]:
df = pd.read_csv("data_for_classifier.csv")

In [3]:
df = df.drop(columns = "datetime")

In [4]:
split_idx = int(len(df) * 0.9)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

X_train = train.drop(columns = "is_fraud")
y_train = train["is_fraud"]

X_test = test.drop(columns = "is_fraud")
y_test = test["is_fraud"]

In [5]:
val_split_idx = int(len(train) * (8/9)) 

X_train_sub = X_train[:val_split_idx]
y_train_sub = y_train[:val_split_idx]

X_val = X_train[val_split_idx:]
y_val = y_train[val_split_idx:]

In [6]:
learning_rates = [0.01, 0.05, 0.1]
max_iters = [100, 200, 500]
max_leaf_nodes = [15, 31, 63]

best_f1 = 0
best_params = {}

for lr, mi, mln in itertools.product(learning_rates, max_iters, max_leaf_nodes):
    gbc = HistGradientBoostingClassifier(
        learning_rate=lr,
        max_iter=mi,
        max_leaf_nodes=mln,
        class_weight="balanced"
    )
    gbc.fit(X_train_sub, y_train_sub)
    y_pred = gbc.predict(X_val)
    f1 = f1_score(y_val, y_pred, pos_label=1)
    if f1 > best_f1:
        best_f1 = f1
        best_params = {"learning_rate" : lr, "max_iter" : mi, "max_leaf_nodes" : mln}

print(f"Best f1 score: {best_f1}")

Best f1 score: 0.9090909090909091


In [11]:
gbc = HistGradientBoostingClassifier(
    learning_rate=best_params["learning_rate"],
    max_iter=best_params["max_iter"],
    max_leaf_nodes=best_params["max_leaf_nodes"],
    class_weight="balanced"
)
gbc.fit(X_train, y_train)
y_pred = gbc.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    108833
           1       0.85      0.96      0.90       185

    accuracy                           1.00    109018
   macro avg       0.92      0.98      0.95    109018
weighted avg       1.00      1.00      1.00    109018



In [12]:
result = permutation_importance(gbc, X_test, y_test, n_jobs=-1)
importance_df = pd.DataFrame({"Feature": X_test.columns, "Importance": result.importances_mean})
importance_df = importance_df.sort_values(by="Importance", ascending=True)
importance_df.to_csv("importance_features_HGBC.csv", index=False)

Recall is more important as high recall means we rarely miss a laundering case (false negative).